# STAI-X Challenge — R sample submission

Minimal, runnable baseline: a **one-dimensional linear regression** that predicts `rate_per_10000_ed_visits` from a single covariate (`gtrends_overdose`).

This notebook demonstrates the submission file format end-to-end:

1. Read the training target + covariates and the validation covariates from `/kaggle/input/staix-challenge/`.
2. Fit `lm(rate ~ gtrends_overdose)` on training rows in the three scoring categories (`all_drugs`, `all_opioids`, `all_stimulants`).
3. Predict for every row in `sample_submission.csv`.
4. Write `/kaggle/working/submission.csv` with **only** `row_id` and `rate_per_10000_ed_visits`.

Use this as a template — replace the model with whatever you like.

In [ ]:
suppressPackageStartupMessages({
  library(readr)
  library(dplyr)
})

DATA <- "/kaggle/input/staix-challenge"

train_y <- read_csv(file.path(DATA, "train/dose_sys_train.csv"), show_col_types = FALSE)
train_x <- read_csv(file.path(DATA, "train/covariates.csv"),     show_col_types = FALSE)
val_x   <- read_csv(file.path(DATA, "val/covariates.csv"),       show_col_types = FALSE)
sub     <- read_csv(file.path(DATA, "sample_submission.csv"),    show_col_types = FALSE)

cat("train_y:", dim(train_y), "| train_x:", dim(train_x),
    "| val_x:", dim(val_x), "| sub:", dim(sub), "\n")

In [ ]:
# Keep only the three scoring categories and join the single predictor.
SCORING <- c("all_drugs", "all_opioids", "all_stimulants")
FEATURE <- "gtrends_overdose"

train <- train_y |>
  filter(overdose_category %in% SCORING) |>
  left_join(train_x |> select(period_id, jurisdiction, all_of(FEATURE)),
            by = c("period_id", "jurisdiction")) |>
  filter(!is.na(rate_per_10000_ed_visits), !is.na(.data[[FEATURE]]))

cat("training rows after filter/merge:", nrow(train), "\n")

In [ ]:
# 1-D linear regression: rate ~ gtrends_overdose.
model <- lm(rate_per_10000_ed_visits ~ gtrends_overdose, data = train)
print(coef(model))

In [ ]:
# Attach the predictor to the submission template via (period_id, jurisdiction).
val_merged <- sub |>
  left_join(val_x |> select(period_id, jurisdiction, all_of(FEATURE)),
            by = c("period_id", "jurisdiction"))

# Fill any missing predictor values with the training mean so predictions stay finite.
feat_mean <- mean(train[[FEATURE]], na.rm = TRUE)
val_merged[[FEATURE]][is.na(val_merged[[FEATURE]])] <- feat_mean

sub$rate_per_10000_ed_visits <- as.numeric(predict(model, newdata = val_merged))

In [ ]:
# Write the submission with EXACTLY the two required columns.
out <- sub[, c("row_id", "rate_per_10000_ed_visits")]

stopifnot(
  nrow(out) == 918,
  ncol(out) == 2,
  !anyDuplicated(out$row_id),
  all(is.finite(out$rate_per_10000_ed_visits))
)

write_csv(out, "/kaggle/working/submission.csv")
cat("wrote /kaggle/working/submission.csv", dim(out), "\n")
head(out)